In [1]:
import pandas as pd
import numpy as np



In [2]:
zomato=pd.read_csv("../data/raw/zomato.csv",encoding="latin-1")
country=pd.read_excel("../data/raw/Country-Code.xlsx")
delivery=pd.read_csv("../data/raw/Food_Delivery_Times.csv")


In [3]:
#Standardizing column names
def clean_cols(df):
    df.columns=[c.strip().lower().replace(" ","_").replace("-","_") for c in df.columns]
    return df

zomato=clean_cols(zomato)
country=clean_cols(country)
delivery=clean_cols(delivery)

print("Zomato columns:",zomato.columns.tolist())
print("Country columns:",country.columns.tolist())



Zomato columns: ['restaurant_id', 'restaurant_name', 'country_code', 'city', 'address', 'locality', 'locality_verbose', 'longitude', 'latitude', 'cuisines', 'average_cost_for_two', 'currency', 'has_table_booking', 'has_online_delivery', 'is_delivering_now', 'switch_to_order_menu', 'price_range', 'aggregate_rating', 'rating_color', 'rating_text', 'votes']
Country columns: ['country_code', 'country']


In [4]:
#Merge country names
zomato=zomato.merge(country,on="country_code",how="left")
zomato=zomato.rename(columns={"country":"country_name"})

In [5]:
#filter India
zomato_india=zomato[zomato["country_name"]=="India"].copy()
print("\nAvailable citites :\n",zomato_india["city"].value_counts().head(20))


Available citites :
 city
New Delhi       5473
Gurgaon         1118
Noida           1080
Faridabad        251
Ghaziabad         25
Ahmedabad         21
Amritsar          21
Bhubaneshwar      21
Guwahati          21
Lucknow           21
Agra              20
Allahabad         20
Aurangabad        20
Bangalore         20
Bhopal            20
Chennai           20
Coimbatore        20
Dehradun          20
Goa               20
Indore            20
Name: count, dtype: int64


In [6]:
#Filter city
city_name="Agra"
city_df=zomato_india[zomato_india["city"].str.lower()==city_name.lower()].copy()
print(f"\n{city_name} restaurants found: { len(city_df)}")



Agra restaurants found: 20


In [7]:
for col in ["aggregate_rating","votes","average_cost_for_two"]:
    city_df[col]=pd.to_numeric(city_df[col],errors="coerce")

In [8]:
#Drop missing values
city_df=city_df.dropna(subset=["restaurant_name","locality","cuisines","aggregate_rating","average_cost_for_two"])

In [9]:
#Remove Duplicates
city_df=city_df.drop_duplicates(subset=["restaurant_id"])

In [10]:
#Save proccessed data
city_df.to_csv("../data/processed/zomato_clean.csv",index=False)
print("saved:Zomato_clean.csv")

saved:Zomato_clean.csv


In [11]:
#Clean Delivery data
for col in ["distance_km","preparation_time_min","courier_experience_yrs","delivery_time_min"]:
    if col in delivery.columns:
        delivery[col] = pd.to_numeric(delivery[col], errors="coerce")

delivery_clean = delivery.dropna().copy()
delivery_clean.to_csv("../data/processed/delivery_clean.csv", index=False)
print(f"Saved: delivery_clean.csv — {len(delivery_clean)} rows")

Saved: delivery_clean.csv — 883 rows
